# 01 — Hello, Orqest Agent 👋

Goal: get a *minimal* agent running using your `BaseAgent`, a tiny `GlobalState`, and a deterministic `EchoAgent` that doesn't require an LLM. We'll build up complexity in later notebooks.

**What you'll learn**
1. How to define an output `BaseModel` for your agent
2. How to subclass `BaseAgent` and implement `_run_implementation`
3. How to run the agent with a `GlobalState`
4. How to register a basic hook at runtime

> This notebook assumes your package is importable (e.g. the repo root is on `PYTHONPATH`). If not, add it in the next cell.

In [2]:
# If needed, uncomment and edit the path to your repo root so `orqest` is importable
# import sys, os
# repo_root = os.path.abspath('..')  # or the absolute path to your repo
# if repo_root not in sys.path:
#     sys.path.insert(0, repo_root)
print('Environment ready ✅')

Environment ready ✅


## 1) Define an output model
Your `BaseAgent` expects an `output_type` that is a Pydantic model. We'll start with a minimal one.

In [3]:
from pydantic import BaseModel, Field
from typing import Optional

class EchoOutput(BaseModel):
    """Deterministic output for Hello World demo."""
    text: str = Field(description="Agent reply text")
    reason: Optional[str] = Field(default=None, description="Why this reply was produced")

EchoOutput(text='hello', reason='demo')

EchoOutput(text='hello', reason='demo')

## 2) Subclass `BaseAgent`
For the first demo, we avoid the LLM entirely and just echo the latest user message from `GlobalState`. This lets you validate your lifecycle and hooks without depending on a model.

In [4]:
import asyncio
from orqest.agents.base_agent import BaseAgent  # adjust import if your path differs
from orqest.agents.hooks import HookPoint
from orqest.agents.state import GlobalState

class EchoAgent(BaseAgent[EchoOutput]):
    async def _run_implementation(self, state: GlobalState, **kwargs) -> EchoOutput:
        # Grab latest user message; fallback if none
        user_msg = state.get_latest_user_message() or "(no user message)"
        # In a real agent, you'd call self.agent / LLM here. We keep it deterministic.
        return EchoOutput(text=user_msg, reason="Echoed latest user message deterministically")

    async def _process_response_implementation(self, response, state: GlobalState, **kwargs) -> EchoOutput:
        # This demo doesn't use a separate processing step; just return the response as-is
        return response

# Instantiate the agent
echo = EchoAgent(
    agent_name="echo",
    system_prompt="You are a simple echo agent.",
    output_type=EchoOutput,
)

## 3) Run with a `GlobalState`
We'll create a tiny state, add a user message, and run the agent.

In [6]:
state = GlobalState()
state.add_message("user", "Hello agent! Can you hear me?")

result = await echo.run(state)
result

GlobalState(messages=[{'role': 'user', 'content': 'Hello agent! Can you hear me?'}], plan=[], chat_history=[])

## 4) Add a runtime hook
Hooks let you inject logic without modifying the agent. We'll add a PRE_RUN hook that appends a tag to the last user message before the agent runs.

In [ ]:
from copy import deepcopy

async def tag_user_message(state: GlobalState, **kwargs):
    # Make a shallow copy to show that hooks can transform state safely
    new_state = deepcopy(state)
    last = new_state.get_latest_user_message()
    if last:
        new_state.add_message("system", "[hook] PRE_RUN tagged the last user message")
        new_state.add_message("user", f"{last} [tagged]")
    return new_state

# Register the hook (higher priority runs first if multiple)
echo.add_hook(HookPoint.PRE_RUN, tag_user_message, priority=10, name="tagger")

state2 = GlobalState()
state2.add_message("user", "Second run, please echo me too")
asyncio.run(echo.run(state2))

### What to try next
- Add a `POST_RUN` hook that uppercases `EchoOutput.text`
- Create a `Middleware` subclass with `pre_run` and `post_run` methods and register via `use_middleware()`
- Swap the deterministic body with a real LLM call using `self.agent` when your model plumbing is ready

In the next notebook, we'll introduce middleware and show how one object can participate in multiple hook points.